# Lab 21: Calibrated scoring versus an LLM asked for a probability

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Project-Lantern-Case-Study-Lab2-Team-2 | LLM mode: replay


In [2]:
# From the slide: Calibrating with scikit-learn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import (
    CalibratedClassifierCV, calibration_curve)
from sklearn.metrics import brier_score_loss, roc_auc_score
log = pd.read_csv("data/reference/deal_log.csv")
FEATURES = ["revenue_usd_m", "growth", "ebitda_margin",
            "recurring_share", "top_customer_share",
            "asking_ev_ebitda"]
X_train, X_test, y_train, y_test, _, p_true_test = \
    train_test_split(log[FEATURES], log["advanced_to_ic"],
        log["p_true_advanced_to_ic"], test_size=0.3,
        random_state=7, stratify=log["advanced_to_ic"])
clf = CalibratedClassifierCV(
    estimator=LogisticRegression(max_iter=1000),
    method="isotonic", cv=5).fit(X_train, y_train)
p = clf.predict_proba(X_test)[:, 1]
print("AUC  ", roc_auc_score(y_test, p))
print("Brier", brier_score_loss(y_test, p))
print("Brier of the TRUE probs",
      brier_score_loss(y_test, p_true_test))  # floor
frac_pos, mean_pred = calibration_curve(
    y_test, p, n_bins=10)                   # the diagram

AUC   0.681842672413793
Brier 0.20523255553773348
Brier of the TRUE probs 0.18106188622222225


In [3]:
import numpy as np, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss, roc_auc_score
log = sd.deal_log(); ids = list(X_test.index)        # the identical test set for both approaches
rows = [{"deal_id": log.loc[i, "deal_id"], **{c: float(X_test.loc[i, c]) for c in X_test.columns}} for i in ids]
llm_scores = {s.deal_id: s.p for s in llm.structured(schemas.Scores, prompts.scorer_messages(rows), llm.P["llm"]["models"]["large"]).scores}
p_llm = np.array([llm_scores[log.loc[i, "deal_id"]] for i in ids]); p_mod = clf.predict_proba(X_test.loc[ids])[:, 1]; yt = y_test.loc[ids]
for name, p in (("calibrated model", p_mod), ("LLM-as-scorer", p_llm)):
    print(f"{name:16s} Brier {brier_score_loss(yt, p):.3f}  AUC {roc_auc_score(yt, p):.3f}  distinct values {len(set(np.round(p, 3)))}")


calibrated model Brier 0.205  AUC 0.682  distinct values 56
LLM-as-scorer    Brier 0.211  AUC 0.721  distinct values 11


Brier score measures squared probability error and reflects both calibration and discrimination; lower is better. AUC measures ranking, not probability calibration.

**Exercise 21.** Count the LLM probability values, save both reliability curves, and bootstrap the paired AUC difference. Decide which approach should supply probabilities and whether the evidence supports a different choice for ranking.

In [4]:
from collections import Counter
counts = sorted(Counter(float(x) for x in np.round(p_llm, 2)).items())
probability_counts = pd.DataFrame(counts, columns=["LLM probability", "count"])
assert probability_counts["count"].sum() == len(yt)
print("LLM probabilities rounded to two decimals:")
display(probability_counts)
print("Distinct rounded probabilities:", len(counts), "| test deals:", len(yt))

from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(6, 5))

for name, probabilities in [
    ("Calibrated model", p_mod),
    ("LLM", p_llm),
]:
    observed, predicted = calibration_curve(
        yt, probabilities, n_bins=10, strategy="uniform"
    )
    ax.plot(predicted, observed, "o-", label=name)

ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax.set_xlabel("Average predicted probability")
ax.set_ylabel("Fraction that actually advanced")
ax.set_title("Predicted probabilities versus observed outcomes")
ax.legend()
ax.grid(alpha=0.3)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
fig.tight_layout()
Path("reports/plots").mkdir(parents=True, exist_ok=True)
fig.savefig("reports/plots/reliability.png", dpi=150, bbox_inches="tight")
print("Saved both model curves to reports/plots/reliability.png")
# Display explicitly because this notebook uses the Agg backend.
from IPython.display import display
display(fig)
plt.close(fig)

LLM probabilities rounded to two decimals:


,LLM probability,count
0,0.05,10
1,0.10,3
2,0.20,8
3,0.30,13
4,0.40,8
5,0.50,16
6,0.60,13
7,0.70,9
8,0.80,3
9,0.90,3


Distinct rounded probabilities: 11 | test deals: 90
Saved both model curves to reports/plots/reliability.png


<Figure size 600x500 with 1 Axes>

In [5]:
# Paired bootstrap: resample the same test deals for both methods.
rng = np.random.default_rng(7)
yv = yt.to_numpy()
assert len(yv) == len(p_llm) == len(p_mod)
assert np.isfinite(p_llm).all() and np.isfinite(p_mod).all()
assert ((0 <= p_llm) & (p_llm <= 1)).all()
assert ((0 <= p_mod) & (p_mod <= 1)).all()
differences, skipped = [], 0
for _ in range(2000):
    idx = rng.integers(0, len(yv), len(yv))
    if np.unique(yv[idx]).size < 2:
        skipped += 1  # AUC is undefined for single-class samples.
        continue
    differences.append(
        roc_auc_score(yv[idx], p_llm[idx]) - roc_auc_score(yv[idx], p_mod[idx])
    )
assert differences
lo, hi = np.percentile(differences, [2.5, 97.5])
auc_llm = roc_auc_score(yv, p_llm)
auc_model = roc_auc_score(yv, p_mod)
print(f"AUC difference (LLM minus calibrated model): {auc_llm - auc_model:.3f}")
print(f"Paired bootstrap 95% interval: [{lo:.3f}, {hi:.3f}]")
print(f"Valid replicates: {len(differences)}; single-class replicates skipped: {skipped}")
print("Interval includes zero:", bool(lo <= 0 <= hi))

report = {
    "test_deals": len(yv),
    "probability_counts": [{"probability": p, "count": count} for p, count in counts],
    "calibrated_model": {"auc": float(auc_model), "brier": float(brier_score_loss(yv, p_mod))},
    "llm": {"auc": float(auc_llm), "brier": float(brier_score_loss(yv, p_llm))},
    "auc_difference_llm_minus_model": float(auc_llm - auc_model),
    "paired_bootstrap": {"seed": 7, "attempted": 2000, "valid": len(differences),
                         "skipped_single_class": skipped, "ci95": [float(lo), float(hi)]},
}
Path("reports/exercise21_scoring.json").write_text(json.dumps(report, indent=2) + "\n")


AUC difference (LLM minus calibrated model): 0.039
Paired bootstrap 95% interval: [-0.111, 0.209]
Valid replicates: 2000; single-class replicates skipped: 0
Interval includes zero: True


1095

**Interpretation.** Round-number predictions concentrate deals at a small set of probability levels. A reliability diagram therefore has fewer distinct occupied regions, with some bins omitted and others averaging many deals; the connected line is not evidence of smooth probability resolution. Inspect the counts alongside the curve.

For agent probabilities, prefer the calibrated model on this sample: it has lower observed Brier error and an explicit calibration procedure. This does not establish a statistically significant Brier improvement or perfect calibration; validate it on fresh deals.

For ranking, the LLM has a higher point-estimate AUC, but the paired bootstrap interval for the difference includes zero. This sample does not establish that either ranks better. LLM judgments may still be useful as features in a calibrated model, provided they are generated without label leakage and evaluated on held-out data.

The bootstrap keeps the fitted model and recorded LLM scores fixed, and resamples test deals in pairs. It measures test-sample uncertainty conditional on this split; it does not capture retraining variability or future distribution shifts.